# Unsupervised Learning Methods in TopologicPy and the Role of Graphs

In [1]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## Import the needed libraries

In [2]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## Check the TopologicPy Version

In [ ]:
print(Helper.Version())

## Utility Function to Reset the Face Dictionaries

In [3]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts[str(value)]
            f = Topology.SetDictionary(f, d)


## 1. Import the gallery floor plan

In [4]:
gallery = Topology.ByBREPPath(r"C:\Users\sarwj\OneDrive - Cardiff University\Documents\GitHub\topologicpy\assets\MachineLearning\gallery.brep")

In [ ]:
Topology.Show(gallery, camera=[0,0,6], faceColor=[210,210,250], faceOpacity=1, edgeColor="white", edgeWidth=3, showVertices=False, backgroundColor="black", width=800, height=600)

## 2. Create a grid overlay

In [5]:
b_r = Wire.BoundingRectangle(gallery)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")
print(width, length)
uRange = list(range(0,int(width)+3,3))
vRange = list(range(0,int(length)+3,3))

grid = Grid.EdgesByDistances(gallery, clip=True, uRange=uRange, vRange=vRange)

214.505355 141.063941


In [ ]:
Topology.Show(gallery, grid, camera=[0,0,6], faceColor=[210,210,250], edgeColor="grey", edgeWidth=3, faceOpacity=1, showVertices=False, backgroundColor="black", width=800, height=600)

## 3. Slice the floor plan with the grid to create a topologic shell

In [6]:
shell = Topology.Slice(gallery, grid)
faces = Topology.Faces(shell)
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_"+str(i+1))
    f = Topology.SetDictionary(f, d)


In [ ]:
Topology.Show(shell, camera=[0,0,6], faceColor=[210,210,250], edgeColor="black", edgeWidth=3, faceOpacity=0.6, showVertices=False, backgroundColor="black", width=800, height=600)

## 4. Derive navigation and analysis graphs from the shell

In [7]:
navigation_graph = Graph.ByTopology(shell, direct=False, viaSharedTopologies=True)
analysis_graph = Graph.ByTopology(shell)

In [8]:
g_verts = Graph.Vertices(analysis_graph)

In [ ]:
Topology.Show(analysis_graph, camera=[0,0,6], vertexSize=4, vertexColor="red", edgeColor="lightgrey", backgroundColor="black", width=800, height=600)

## 5. Analyse Graph: (Examples: Shortest Path and Closeness Centrality/Integration)

### Shortest Path (Use navigation graph)

In [9]:
import time

start_vertex = Vertex.ByCoordinates(xmin+2, ymax-2,0) # Upper left corner
end_vertex = Vertex.ByCoordinates(xmax-2,ymin+2,0) # Lower right corner

crg = Graph.CompiledRoutingGraph(navigation_graph, precomputeTurns=True)
path1 = Graph.ShortestPath(crg, vertexA=start_vertex, vertexB=end_vertex)
for i in range(4):
    start = time.time()
    shortest_path = Graph.ShortestPath(crg, vertexA=start_vertex, vertexB=end_vertex)
    end = time.time()
    print("Duration:", round(end-start, 2))

# straight_path = Wire.Straighten(shortest_path, host=gallery)
# print("Original Path Length:", round(Wire.Length(shortest_path), 2))
# print("Straightened Path Length:", round(Wire.Length(straight_path), 2))
# edges = Topology.Edges(shortest_path)
# for edge in edges:
#     d = Dictionary.ByKeysValues(["width", "color"], [7, "red"])
#     edge = Topology.SetDictionary(edge, d)
# edges = Topology.Edges(straight_path)
# for edge in edges:
#     d = Dictionary.ByKeysValues(["width", "color"], [7, "blue"])
#     edge = Topology.SetDictionary(edge, d)

Duration: 2.03
Duration: 2.02
Duration: 2.07
Duration: 2.1


In [ ]:
Topology.Show(shell, shortest_path, straight_path, camera=[0,0,6], faceColor=[210,210,250], edgeColorKey="color", edgeWidthKey="width", backgroundColor="black", width=800, height=600)

### Closeness Centrality/Integration
* Closeness centrality is a graph metric that quantifies how close a node is to all other nodes by taking the reciprocal of the sum of its shortest path distances to every other node in the network.
* In space syntax, closeness centrality corresponds to global integration, measuring how spatially accessible or topologically shallow a space is within a configuration, thereby indicating its potential for movement flow and encounter density.

In [ ]:
centrality_list = Graph.ClosenessCentrality(analysis_graph, colorScale="thermal")

## 6. Transfer the information from the graph back to the shell

In [ ]:
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [ ]:
Topology.Show(faces, faceColorKey="cc_color", faceOpacity=1, showEdges=False, showVertices=False, camera=[0,0,6], backgroundColor="black", width=800, height=600)

## K-Means
* K-means clustering is an unsupervised learning algorithm that partitions a dataset of n observations into K disjoint clusters by minimizing the within-cluster sum of squared distances (WCSS) between data points and their assigned cluster centroid.

In [ ]:
k = 35
clusters = Cluster.KMeans(g_verts, k=k, keys=["x", "y"])
colors = [Color.ByValueInRange(x, minValue=0, maxValue=k-1, colorScale="thermal") for x in range(k)]
for i, cluster in enumerate(clusters):
    color = colors[i]
    c_verts = Topology.Vertices(cluster)
    for c_vert in c_verts:
        d = Topology.Dictionary(c_vert)
        d = Dictionary.SetValuesAtKeys(d, ["km_id", "km_color"], [i, color])
        c_vert = Topology.SetDictionary(c_vert, d)

### 6. Transfer the information from the grid back to the shell

In [ ]:
reset_dictionaries(shell)
shell = Topology.TransferDictionariesBySelectors(shell, selectors=g_verts, tranFaces=True)

In [ ]:
Topology.Show(shell, faceColorKey="km_color", faceOpacity=1, edgeWidthKey="width", showVertices=False, camera=[0,0,5], backgroundColor="black", width=800, height=600)

## DBSCAN
* DBSCAN (Density-Based Spatial Clustering of Applications with Noise) is an unsupervised clustering algorithm that groups points based on local density rather than distance to a centroid. It defines clusters as contiguous regions of high point density separated by regions of low density.

In [ ]:
spiral_1 = Wire.Spiral(radiusA=1, radiusB=5, height=10, turns=2)
verts = []
for i in range(0, 101, 1):
    v = Wire.VertexByParameter(spiral_1, i*0.01)
    verts.append(v)
spiral_2 = Topology.Rotate(spiral_1, angle=120)
for i in range(0, 101, 1):
    v = Wire.VertexByParameter(spiral_2, i*0.01)
    verts.append(v)
spiral_3 = Topology.Rotate(spiral_1, angle=240)
for i in range(0, 101, 1):
    v = Wire.VertexByParameter(spiral_3, i*0.01)
    verts.append(v)
Topology.Show(verts, vertexSize=12, backgroundColor="white", width=500, height=500)

In [ ]:
dis = Vertex.Distance(verts[0], verts[1])
result = Cluster.DBSCAN(verts, keys=["x", "y", "z"], epsilon=dis*1.1, minSamples=2)
clusters = result[0]
print("Final Number of Clusters:", len(clusters))
colors = ["red", "blue", "green"]
for i, cluster in enumerate(clusters):
    color = colors[i]
    c_verts = Topology.Vertices(cluster)
    for c_vert in c_verts:
        d = Topology.Dictionary(c_vert)
        d = Dictionary.SetValueAtKey(d, "db_color", color)
        c_vert = Topology.SetDictionary(c_vert, d)
Topology.Show(verts, vertexColorKey="db_color", vertexSize=14, camera=[2,0,0.5], backgroundColor="white", width=500, height=500)

## 5. Analyse the graph

In [ ]:
n_clusters = 0
attempts = 1
epsilon = 1
while n_clusters < 15 and attempts < 20:
    result = Cluster.DBSCAN(g_verts, keys=["x", "y"], epsilon=epsilon, minSamples=10)
    clusters = result[0]
    n_clusters = len(clusters)
    print("Attempt:", attempts)
    print("  Epsilon:", round(epsilon, 2), "---> Clusters:", n_clusters)
    attempts += 1
    epsilon += 1
print("Final Number of Clusters:", len(clusters))

In [ ]:
colors = [Color.ByValueInRange(x, minValue=0, maxValue=len(clusters)-1, colorScale="thermal") for x in range(len(clusters))]
for i, cluster in enumerate(clusters):
    color = colors[i]
    c_verts = Topology.Vertices(cluster)
    for c_vert in c_verts:
        d = Topology.Dictionary(c_vert)
        d = Dictionary.SetValuesAtKeys(d, ["db_id", "db_color"], [i, color])
        c_vert = Topology.SetDictionary(c_vert, d)

## 6. Transfer the information from the graph back to the shell

In [ ]:
reset_dictionaries(shell)
shell = Topology.TransferDictionariesBySelectors(shell, selectors=g_verts, tranFaces=True)

In [ ]:
Topology.Show(shell, faceColorKey="db_color", faceOpacity=1, edgeWidthKey="width", showVertices=False, camera=[0,0,5], backgroundColor="black", width=800, height=600)

## HDBSCAN
* HDBSCAN (Hierarchical Density-Based Spatial Clustering of Applications with Noise) extends DBSCAN by building a cluster hierarchy and extracting stable clusters automatically.
* Unlike DBSCAN, HDBSCAN does **not** require an epsilon parameter, making it more robust for datasets with clusters of varying density.
* It works by computing mutual reachability distances, building a minimum spanning tree, constructing a condensed cluster tree, and then selecting the most stable clusters.

### HDBSCAN on Spiral Data

In [ ]:
# Reload the module to pick up our new HDBSCAN method
import importlib
import topologicpy.Cluster
importlib.reload(topologicpy.Cluster)
from topologicpy.Cluster import Cluster

hdb_result = Cluster.HDBSCAN(verts, keys=["x", "y", "z"], minClusterSize=5)
hdb_clusters = hdb_result[0]
hdb_noise = hdb_result[1]
print("HDBSCAN - Number of Clusters:", len(hdb_clusters))
if hdb_noise:
    print("HDBSCAN - Number of Noise Points:", len(Topology.Vertices(hdb_noise)))
else:
    print("HDBSCAN - Number of Noise Points: 0")

hdb_colors = ["red", "blue", "green", "orange", "purple", "cyan", "magenta", "yellow"]
for i, hdb_cluster in enumerate(hdb_clusters):
    hdb_color = hdb_colors[i % len(hdb_colors)]
    hdb_c_verts = Topology.Vertices(hdb_cluster)
    for hdb_c_vert in hdb_c_verts:
        d = Topology.Dictionary(hdb_c_vert)
        d = Dictionary.SetValueAtKey(d, "hdb_color", hdb_color)
        hdb_c_vert = Topology.SetDictionary(hdb_c_vert, d)
Topology.Show(verts, vertexColorKey="hdb_color", vertexSize=14, camera=[2,0,0.5], backgroundColor="white", width=500, height=500)

### HDBSCAN on Gallery Floor Plan

## 5. Analyse the graph

In [ ]:
hdb_result = Cluster.HDBSCAN(g_verts, keys=["x", "y"], minClusterSize=15)
hdb_clusters = hdb_result[0]
hdb_noise = hdb_result[1]
print("HDBSCAN - Number of Clusters:", len(hdb_clusters))
if hdb_noise:
    print("HDBSCAN - Number of Noise Points:", len(Topology.Vertices(hdb_noise)))
else:
    print("HDBSCAN - Number of Noise Points: 0")

In [ ]:
hdb_colors = [Color.ByValueInRange(x, minValue=0, maxValue=max(len(hdb_clusters)-1, 1), colorScale="thermal") for x in range(len(hdb_clusters))]
for i, hdb_cluster in enumerate(hdb_clusters):
    hdb_color = hdb_colors[i]
    hdb_c_verts = Topology.Vertices(hdb_cluster)
    for hdb_c_vert in hdb_c_verts:
        d = Topology.Dictionary(hdb_c_vert)
        d = Dictionary.SetValuesAtKeys(d, ["hdb_id", "hdb_color"], [i, hdb_color])
        hdb_c_vert = Topology.SetDictionary(hdb_c_vert, d)

## 6. Transfer the information from the graph back to the shell

In [ ]:
reset_dictionaries(shell)
shell = Topology.TransferDictionariesBySelectors(shell, selectors=g_verts, tranFaces=True)

In [ ]:
Topology.Show(shell, faceColorKey="hdb_color", faceOpacity=1, edgeWidthKey="width", showVertices=False, camera=[0,0,5], backgroundColor="black", width=800, height=600)

## DBSCAN vs HDBSCAN Comparison
| Feature | DBSCAN | HDBSCAN |
|---------|--------|--------|
| **Epsilon parameter** | Required (sensitive to choice) | Not needed |
| **Varying density** | Struggles with clusters of different densities | Handles varying density naturally |
| **Cluster count** | Depends on epsilon | Automatically determined |
| **Algorithm** | Single density threshold | Hierarchical density-based extraction |
| **Noise handling** | Points below density threshold | Points that never join stable clusters |

In [ ]:
# Compare DBSCAN and HDBSCAN results on the gallery floor plan
db_result = Cluster.DBSCAN(g_verts, keys=["x", "y"], epsilon=5, minSamples=10)
db_clusters_cmp = db_result[0]
db_noise_cmp = db_result[1]

hdb_result_cmp = Cluster.HDBSCAN(g_verts, keys=["x", "y"], minClusterSize=15)
hdb_clusters_cmp = hdb_result_cmp[0]
hdb_noise_cmp = hdb_result_cmp[1]

print("=== DBSCAN (epsilon=5, minSamples=10) ===")
print(f"  Clusters: {len(db_clusters_cmp)}")
print(f"  Noise points: {len(Topology.Vertices(db_noise_cmp)) if db_noise_cmp else 0}")

print()
print("=== HDBSCAN (minClusterSize=15) ===")
print(f"  Clusters: {len(hdb_clusters_cmp)}")
print(f"  Noise points: {len(Topology.Vertices(hdb_noise_cmp)) if hdb_noise_cmp else 0}")

print()
print("Key difference: HDBSCAN found clusters without needing to specify an epsilon radius.")

## Graph Community Partition (Louvain)
### Graph community partition using the Louvain method is an unsupervised algorithm for detecting communities in a network by maximising modularity, a measure of the density of edges within groups compared to edges between groups.

## 5. Analyse the graph

In [ ]:
_ = Graph.CommunityPartition(analysis_graph, colorScale="thermal")

## 6. Transfer the information from the graph back to the shell

In [ ]:
reset_dictionaries(shell)
shell = Topology.TransferDictionariesBySelectors(shell, selectors=g_verts, tranFaces=True)

In [ ]:
Topology.Show(shell, faceColorKey="cp_color", faceOpacity=1, edgeWidthKey="width", showVertices=False, camera=[0,0,5], backgroundColor="black", width=800, height=600)